# Equation of State

Prepare the energy-volume data needed for an equation-of-state (EOS) study by creating isotropically scaled variants of a material, running a total-energy workflow for each variant on the Mat3ra platform, and plotting energy per volume versus volume.

<h2 style="color:green">Usage</h2>

1. Set material, scaling, workflow, and compute parameters in cell 1.2 below.
1. Click "Run" > "Run All" to create scaled materials and jobs.
1. Wait for all total-energy jobs to complete.
1. Scroll down to review the final energy-volume table and plot.

## Summary

1. Set up the environment and define EOS parameters, including scale factors, workflow, and compute settings.
1. Authenticate and initialize the API client, then select the working account and project.
1. Load a source material, create scaled variants, and save each variant to the platform.
1. Create a total-energy workflow for the selected application.
1. Create the compute configuration for the jobs.
1. Create one total-energy job per scaled material and track the generated job IDs.
1. Submit all jobs and monitor their statuses together.
1. Retrieve total-energy results and plot energy per volume versus volume.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")

### 1.2. Set parameters and configurations for the workflow and job

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = "seminar"

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # Name of the material to load from local file or Standata
SCALE_FACTORS = [0.96, 0.98, 1.00, 1.02]  # Isotropic lattice scaling factors
VISUALIZATION_REPETITIONS = [1, 1, 1]

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "total_energy.json"  # Search term for Workflows Standata
APPLICATION_NAME = "nwchem"  # Can be switched to another supported application
ADD_RELAXATION = None  # Whether to add relaxation subworkflow as first unit
# Set k-grid for relaxation and SCF where supported (if not set, application defaults are used)
RELAXATION_KGRID = None  # e.g., [4, 4, 4]
SCF_KGRID = None  # e.g., [4, 4, 4]

MY_WORKFLOW_NAME = "Equation of State - Total Energy" + (" (relax)" if ADD_RELAXATION else "")

# 5. Compute parameters
CLUSTER_NAME = "001"  # Specify full or partial hostname, e.g. "cluster-001"
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
EOS_JOB_PREFIX = "Equation of State"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from utils.auth import authenticate


await authenticate()

### 2.2. Initialize API Client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({project_id})")

## 3. Create scaled EOS materials
### 3.1. Load source material

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from utils.jupyterlite import load_material_from_folder
from utils.visualize import visualize_materials

base_material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)

visualize_materials(base_material, repetitions=VISUALIZATION_REPETITIONS, title="Source material")

### 3.2. Create scaled material variants

In [ ]:
import pandas as pd

from utils.equation_of_state import build_eos_candidate_records, create_scaled_materials

scaled_materials = create_scaled_materials(base_material, SCALE_FACTORS)
candidate_records = build_eos_candidate_records(scaled_materials, SCALE_FACTORS)
candidates_df = pd.DataFrame(candidate_records)
candidates_df

In [ ]:
preview_materials = [
    {
        "material": material,
        "title": f"scale={scale_factor:.4f}",
        "repetitions": VISUALIZATION_REPETITIONS,
    }
    for scale_factor, material in zip(SCALE_FACTORS, scaled_materials)
]

visualize_materials(preview_materials)

### 3.3. Save scaled materials to the platform

In [ ]:
from utils.api import get_or_create_material
from utils.equation_of_state import get_material_label

saved_materials = []
material_records = []

for scale_factor, scaled_material, candidate_record in zip(SCALE_FACTORS, scaled_materials, candidate_records):
    saved_material_response = get_or_create_material(client, scaled_material, ACCOUNT_ID)
    saved_material = Material.create(saved_material_response)
    saved_materials.append(saved_material)
    material_records.append(
        {
            **candidate_record,
            "material_id": saved_material.id,
            "material_label": get_material_label(saved_material),
        }
    )

materials_df = pd.DataFrame(material_records)
materials_df

## 4. Create workflow and set its parameters
### 4.1. Get list of applications and select one

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Create workflow from standard workflows and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from utils.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Modify workflow (Optional)
#### 4.3.1. Add relaxation

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()

#### 4.3.2. Modify important settings
Set k-grid where the selected workflow supports it.

In [ ]:
from mat3ra.wode.context.providers import PointsGridDataProvider

if RELAXATION_KGRID is not None and ADD_RELAXATION:
    new_context_relax = PointsGridDataProvider(dimensions=RELAXATION_KGRID, isEdited=True).yield_data()
    relaxation_subworkflow = workflow.subworkflows[0]
    unit_to_modify_relax = relaxation_subworkflow.get_unit_by_name(name_regex="relax")
    unit_to_modify_relax.add_context(new_context_relax)
    relaxation_subworkflow.set_unit(unit_to_modify_relax)

if SCF_KGRID is not None:
    new_context_scf = PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data()
    total_energy_subworkflow = workflow.subworkflows[1 if ADD_RELAXATION else 0]
    unit_to_modify_scf = total_energy_subworkflow.get_unit_by_name(name="pw_scf")
    unit_to_modify_scf.add_context(new_context_scf)
    total_energy_subworkflow.set_unit(unit_to_modify_scf)

visualize_workflow(workflow)

### 4.4. Save workflow to collection

In [ ]:
from utils.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN,
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create jobs for all scaled materials
### 6.1. Create one total-energy job per material

In [ ]:
from utils.api import create_jobs_for_materials
from utils.equation_of_state import get_material_label

base_label = get_material_label(base_material)
job_prefix = f"{EOS_JOB_PREFIX} {base_label} {timestamp}"

jobs = create_jobs_for_materials(
    api_client=client,
    materials=saved_materials,
    workflow=saved_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_prefix,
    compute=compute.to_dict(),
    job_name_getter=lambda material, index: f"{job_prefix} | scale={SCALE_FACTORS[index]:.4f}",
)

job_records = []
for material_record, job in zip(material_records, jobs):
    job_records.append(
        {
            **material_record,
            "job_id": job["_id"],
            "job_name": job["name"],
            "status": job.get("status"),
        }
    )

jobs_df = pd.DataFrame(job_records)
jobs_df

## 7. Submit jobs and monitor statuses

In [ ]:
from utils.api import submit_jobs

job_ids = [record["job_id"] for record in job_records]
submit_jobs(client.jobs, job_ids)
print(f"Submitted {len(job_ids)} jobs successfully.")

In [ ]:
job_ids=['dZXf67bBTjA8Pu3zd',
 '6L9m8WXiwphPzrwjw',
 'c983swCLeu64MhACB',
 'bwjNbZ676dsrWzrMD']

In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=POLL_INTERVAL)

## 8. Retrieve total energies and plot
### 8.1. Retrieve total energy properties for all jobs

In [ ]:
from mat3ra.prode import PropertyName

results_records = []
for record in job_records:
    job = client.jobs.get(record["job_id"])
    properties = client.properties.get_for_job(
        record["job_id"], property_name=PropertyName.scalar.total_energy.value
    )
    total_energy = properties[0].get("value") if properties else None
    results_records.append(
        {
            **record,
            "final_status": job.get("status"),
            "total_energy": total_energy,
        }
    )

results_df = pd.DataFrame(results_records).sort_values("volume").reset_index(drop=True)
results_df

### 8.2. Plot energy per volume versus volume

In [ ]:
import plotly.express as px

successful_df = results_df[(results_df["final_status"] == "finished") & results_df["total_energy"].notna()].copy()
plot_df = successful_df.sort_values("volume").copy()
plot_df["energy_per_volume"] = plot_df["total_energy"] / plot_df["volume"]


fig = px.line(
    plot_df,
    x="volume",
    y="energy_per_volume",
    markers=True,
    text="scale_factor",
    hover_data=["job_id", "material_id", "total_energy", "final_status"],
    title="Energy per Volume vs Volume",
)
fig.update_traces(textposition="top center")
fig.update_layout(
    xaxis_title="Volume (Angstrom^3)",
    yaxis_title="Energy / Volume (eV / Angstrom^3)",
)
fig.show()
